In [7]:
import psycopg2

def query_keyword(query_text, k=6):
    conn = psycopg2.connect(
        dbname="mydb",
        user="admin",
        password="1234",
        host="localhost",
        port="5432"
    )
    cur = conn.cursor()

    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_trgm;")
    conn.commit()
    
    sql_query = """
    SELECT content, word_similarity(%s, content) AS keyword_score
    FROM documents
    WHERE content ILIKE %s OR word_similarity(%s, content) > 0.05
    ORDER BY keyword_score DESC
    LIMIT %s;
    """

    like_query = f"%{query_text}%"

    cur.execute(sql_query, (query_text, like_query, query_text, k))
    result = cur.fetchall()
    
    cur.close()
    conn.close()
    return result

result = query_keyword("ปริญญาตรี สาขาเทคโนโลยีคอมพิวเตอร์")
for row in result:
    print(row)

('เปิดหลักสูตรระดับปริญญาตรี 11 สาขา ค.อ.บ. สาขาวิชาครุศาสตร์การออกแบบ ค.อ.บ. สาขาวิชาครุศาสตร์วิศวกรรม ค.อ.บ. สาขาวิชาวิทยาการจัดการเรียนรู้ ค.อ.บ. สาขาวิชาอิเล็กทรอนิกส์และโทรคมนาคม ค.อ.บ. สาขาวิชาเทคโนโลยีคอมพิวเตอร์ ค.อ.บ. สาขาวิชานวัตกรรมและเทคโนโลยีการออกแบบ ทล.บ. สาขาวิชาบูรณาการนวัตกรรมเพื่อสินค้าและบริการ (ต่อเนื่อง) ค.อ.บ. สาขาวิชาครุศาสตร์เกษตร ทล.บ. สาขาวิชาเทคโนโลยีอิเล็กทรอนิกส์ (ต่อเนื่อง) ค.อ.บ. สาขาวิชาการออกแบบสภาพแวดล้อมภายใน ค.อ.บ. สาขาวิชาสถาปัตยกรรม', 0.5897436)
('เปิดหลักสูตรระดับปริญญาโท 3 สาขา ค.อ.ม. การบริหารการศึกษา วท.ม. การศึกษาเกษตร วศ.ม. วิศวกรรมไฟฟ้าและคอมพิวเตอร์ (สหวิทยาการ)', 0.29411766)
('เปิดหลักสูตรระดับปริญญาเอก 4 สาขา ปร.ด. วิศวกรรมไฟฟ้าศึกษา ปร.ด. การศึกษาเกษตร ปร.ด. นวัตกรรมการบริหารการศึกษา (หลักสูตรนานาชาติ) ปร.ด. นวัตเกษตรเขตร้อน (หลักสูตรนานาชาติ)', 0.24390244)
('คณะครุศาสตร์อุตสาหกรรม จัดตั้งเมื่อวันที่ 10 พฤศจิกายน พ.ศ. 2520 จัดตั้งขึ้นเพื่อสนองวัตถุประสงค์ของสถาบันในการผลิตครูอาชีวศึกษาระดับปริญญาตรี และดำเนินการศึกษาและวิจัยด้านเทคโนโลย